# AIMarx TRAIN-06 — Qwen2.5-3B guarded pilot to step 20

Free-tier Tesla T4 only. Recreate checkpoints 1 and 5, inspect both gates, then resume to 20 optimizer steps (one effective epoch over 80 approved train records). No Drive mount, Hub push, paid compute, or smoke-test training.


In [ ]:
import os, pathlib, subprocess, sys, json, hashlib, shutil, zipfile
assert os.path.exists('/content'), 'Run in Google Colab'
subprocess.run(['nvidia-smi'], check=True)


## 1. Checkout the immutable runner and prepare approved data


In [ ]:
REPO = 'https://github.com/hongkhang21998-creator/AIMarx.git'
PINNED_COMMIT = '03bd6c91b3f84960bd7365f382ee13c72d15730f'
repo = pathlib.Path('/content/AIMarx')
if repo.exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=repo, check=True)
else:
    subprocess.run(['git', 'clone', REPO, str(repo)], check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=repo, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip() == PINNED_COMMIT
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'training/colab_qwen25_3b/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.prepare', '/content/aimarx-qwen25-3b-pilot20-data'], check=True)


## 2. Gate 1 — create and inspect checkpoint 1


In [ ]:
DATA = '/content/aimarx-qwen25-3b-pilot20-data'
OUTPUT = '/content/aimarx-qwen25-3b-pilot20-output'
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.train', '--data', DATA, '--output', OUTPUT, '--stop-after', '1'], check=True)
checkpoint_1 = pathlib.Path(OUTPUT) / 'checkpoint-1'
manifest_1 = json.loads((checkpoint_1 / 'aimarx-manifest.json').read_text())
assert manifest_1['global_step'] == 1
print(json.dumps(manifest_1, indent=2))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
APPROVE_RESUME_TO_5 = False
assert APPROVE_RESUME_TO_5 is True, 'STOP: inspect checkpoint-1 and T4 before step 5'
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.train', '--data', DATA, '--output', OUTPUT, '--resume-from', str(checkpoint_1), '--stop-after', '5'], check=True)
checkpoint_5 = pathlib.Path(OUTPUT) / 'checkpoint-5'
manifest_5 = json.loads((checkpoint_5 / 'aimarx-manifest.json').read_text())
assert manifest_5['global_step'] == 5
print(json.dumps(manifest_5, indent=2))
subprocess.run(['nvidia-smi'], check=True)


## 3. Gate 2 — resume checkpoint 5 to one effective epoch

Only continue if checkpoint 5 is complete, no OOM occurred, and the free T4 remains healthy.


In [ ]:
APPROVE_RESUME_TO_20 = False
assert APPROVE_RESUME_TO_20 is True, 'STOP: inspect checkpoint-5 and authorize one effective epoch'
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.train', '--data', DATA, '--output', OUTPUT, '--resume-from', str(checkpoint_5)], check=True)
checkpoint_20 = pathlib.Path(OUTPUT) / 'checkpoint-20'
manifest_20 = json.loads((checkpoint_20 / 'aimarx-manifest.json').read_text())
assert manifest_20['global_step'] == 20
print(json.dumps(manifest_20, indent=2))


## 4. Evaluate validation and package the reproducible result


In [ ]:
evaluation_path = pathlib.Path(OUTPUT) / 'evaluation-step20.json'
subprocess.run([sys.executable, '-m', 'training.colab_qwen25_3b.evaluate', '--data', DATA, '--checkpoint', str(checkpoint_20), '--expected-step', '20', '--output', str(evaluation_path)], check=True)
print(evaluation_path.read_text())
archive = pathlib.Path(shutil.make_archive('/content/AIMarx-Qwen2.5-3B-QLoRA-pilot-step20', 'zip', OUTPUT))
with zipfile.ZipFile(archive) as package:
    names = package.namelist()
    assert all(not pathlib.PurePosixPath(name).is_absolute() and '..' not in pathlib.PurePosixPath(name).parts for name in names)
print(json.dumps({'artifact': archive.name, 'bytes': archive.stat().st_size, 'sha256': hashlib.sha256(archive.read_bytes()).hexdigest(), 'entries': len(names)}, indent=2))
from google.colab import files
files.download(str(archive))
